In [1]:
import matplotlib.pyplot as plt
from statsmodels.base.model import GenericLikelihoodModel
import cv2
import scienceplots
import tifffile as tiff

from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia
from boulder_statistics.analysis.external_data_encyclopedia import ExternalDataEncyclopedia
plt.style.use('science')
plt.rcParams["figure.figsize"] = (3.5, 3.5 * ((5**0.5 - 1) / 2)) # 3.5
plt.rcParams["figure.dpi"] = 600
%matplotlib inline

import polars as pl
from polars import Expr, LazyFrame, DataFrame
import numpy as np
from pathlib import Path
from typing import Any
import tifffile
from typing import Dict
from typing import Callable


from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

dp = DataProductEncyclopedia(
    data_products_path=Path(r"C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\.data_products"))

ed = ExternalDataEncyclopedia(
    external_data_path=Path(r"C:\Users\Joshu\Documents\AO33_backup\Boulder_database\AO33\external_data")
)

In [2]:
exports_folder = Path("dp_export_tests")
areas_export_path = exports_folder / "00_add_areas"
clean_db_export_path: Path = exports_folder / "01_cleanup_db"
add_spectral_export_path: Path = exports_folder / "02_add_spectral"

In [4]:
pl.scan_parquet(clean_db_export_path).filter(pl.col("i") == 2).collect()

face,i,j,row_id,u8_R,32f_R,x,y,z,area,long,lat
enum,u32,u32,u32,u8,f32,f32,f32,f32,f32,f32,f32
"""negx""",2,625,13978,108,0.018777,0.121296,0.143183,-0.143118,0.00074,49.730808,-37.331738
"""negx""",2,645,13978,166,0.02885,0.120656,0.143253,-0.143188,0.000731,49.893871,-37.398129
"""negx""",2,647,13978,167,0.02895,0.120593,0.143257,-0.143193,0.000823,49.909416,-37.404655
"""negx""",2,648,13978,163,0.028212,0.12056,0.143263,-0.143193,0.000889,49.918354,-37.407139
"""negx""",2,610,13978,147,0.025441,0.121759,0.143115,-0.143045,0.000851,49.609577,-37.281113
…,…,…,…,…,…,…,…,…,…,…,…
"""posz""",2,8062,1282073,210,0.037474,-0.136194,-0.131916,-0.136151,0.000706,-135.913986,-35.681034
"""posz""",2,8121,1297706,141,0.024421,-0.135762,-0.133455,-0.135724,0.000646,-135.491028,-35.486649
"""posz""",2,8123,1297706,133,0.023053,-0.135752,-0.133507,-0.135705,0.000665,-135.477783,-35.478565


In [3]:
from boulder_statistics.refinement_plus.bulk_parse_data_tir_maps import DataTirMaps
data_tir_maps: DataFrame = DataTirMaps.bulk_parse(ed)

data_tir_maps

facet_num,latitude,longitude,radius,band depth 350,sigma,band depth 440,slope 1000,ratio 1000
f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0,34.076935,128.249664,0.228102,1.00248,0.001575,null,null,null
1.0,34.080212,128.760391,0.228082,1.002213,0.001462,null,null,null
2.0,34.63451,129.115143,0.227933,1.002295,0.001517,null,null,null
3.0,34.572929,129.757385,0.228036,1.002394,0.001397,null,null,null
4.0,35.244423,130.311279,0.22801,1.002464,0.001438,null,null,null
…,…,…,…,…,…,…,…,…
786427.0,-35.069458,36.039497,0.240551,null,-9999.0,null,null,-9999.0
786428.0,-35.012486,36.149452,0.240428,null,-9999.0,null,null,-9999.0
786429.0,-35.024361,36.241005,0.240386,null,-9999.0,null,null,-9999.0


In [10]:
import polars as pl

N = 8192

B_lf = (
    data_tir_maps
    # degrees -> radians
    .with_columns(
        [
            (pl.col("latitude") * np.pi / 180).alias("phi"),
            (pl.col("longitude") * np.pi / 180).alias("lam"),
        ]
    )
    # sphere coordinates
    .with_columns(
        [
            (pl.col("phi").cos() * pl.col("lam").sin()).alias("sx"),
            pl.col("phi").sin().alias("sy"),
            (pl.col("phi").cos() * pl.col("lam").cos()).alias("sz"),
        ]
    )
    # determine face
    .with_columns(
        [
            pl.when(
                (pl.col("sx").abs() >= pl.col("sy").abs()) &
                (pl.col("sx").abs() >= pl.col("sz").abs()) &
                (pl.col("sx") >= 0)
            ).then(pl.lit("posx"))

            .when(
                (pl.col("sx").abs() >= pl.col("sy").abs()) &
                (pl.col("sx").abs() >= pl.col("sz").abs()) &
                (pl.col("sx") < 0)
            ).then(pl.lit("negx"))

            .when(
                (pl.col("sy").abs() >= pl.col("sx").abs()) &
                (pl.col("sy").abs() >= pl.col("sz").abs()) &
                (pl.col("sy") >= 0)
            ).then(pl.lit("posy"))

            .when(
                (pl.col("sy").abs() >= pl.col("sx").abs()) &
                (pl.col("sy").abs() >= pl.col("sz").abs()) &
                (pl.col("sy") < 0)
            ).then(pl.lit("negy"))

            .when(
                (pl.col("sz").abs() >= pl.col("sx").abs()) &
                (pl.col("sz").abs() >= pl.col("sy").abs()) &
                (pl.col("sz") >= 0)
            ).then(pl.lit("posz"))

            .otherwise(pl.lit("negz"))
            .alias("face")
        ]
    )
)

B_lf = B_lf.with_columns(
    [
        pl.when(pl.col("face") == "posx")
        .then((1 - pl.col("sz")) / 2)

        .when(pl.col("face") == "negx")
        .then((pl.col("sz") + 1) / 2)

        .when(pl.col("face") == "posy")
        .then((pl.col("sx") + 1) / 2)

        .when(pl.col("face") == "negy")
        .then((pl.col("sx") + 1) / 2)

        .when(pl.col("face") == "posz")
        .then((pl.col("sx") + 1) / 2)

        .otherwise((1 - pl.col("sx")) / 2)
        .alias("u"),

        pl.when(pl.col("face").is_in(["posx", "negx"]))
        .then((pl.col("sy") + 1) / 2)

        .when(pl.col("face") == "posy")
        .then((1 - pl.col("sz")) / 2)

        .when(pl.col("face") == "negy")
        .then((pl.col("sz") + 1) / 2)

        .when(pl.col("face") == "posz")
        .then((pl.col("sy") + 1) / 2)

        .otherwise((pl.col("sy") + 1) / 2)
        .alias("v"),
    ]
)

B_lf = B_lf.with_columns(
    [
        (pl.col("u") * N).floor().cast(pl.UInt32).alias("i"),
        (pl.col("v") * N).floor().cast(pl.UInt32).alias("j"),
    ]
)

B_lf.filter(pl.col("i") == 6196, pl.col("j") == 6391, pl.col("face") == "posx")

facet_num,latitude,longitude,radius,band depth 350,sigma,band depth 440,slope 1000,ratio 1000,phi,lam,sx,sy,sz,face,u,v,i,j
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,u32,u32
0.0,34.076935,128.249664,0.228102,1.00248,0.001575,null,null,null,0.594755,2.238379,0.65047,0.560306,-0.512783,"""posx""",0.756392,0.780153,6196,6391
0.0,34.076935,128.249664,0.228102,null,0.000752,1.01098,null,null,0.594755,2.238379,0.65047,0.560306,-0.512783,"""posx""",0.756392,0.780153,6196,6391
0.0,34.076935,128.249664,0.228102,null,0.000838,null,1.001388,null,0.594755,2.238379,0.65047,0.560306,-0.512783,"""posx""",0.756392,0.780153,6196,6391


In [14]:
pl.scan_parquet(clean_db_export_path).explain()

'Parquet SCAN [dp_export_tests\\01_cleanup_db]\nPROJECT */12 COLUMNS\nESTIMATED ROWS: 134519709'